# Fine-tuning YOLOv8 — Mejora detección de chaleco (EPP_merged_v2)
**Materia:** Inteligencia Artificial en Sistemas Embebidos — 2026

Parte del modelo `modelo_epp_v3/best.pt` ya entrenado y agrega el dataset Safety Vests
para mejorar la clase `no-vest`.

**Dataset:** EPP_merged_v2 (~32K imágenes base + 3,491 de Safety Vests)

### Antes de ejecutar:
1. Ir a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**
2. Subir `EPP_merged_v2.zip` a Google Drive en la carpeta `EPP_Dataset`
3. Asegurarte de que `modelo_epp_v3/weights/best.pt` sigue en Drive (del entrenamiento anterior)
4. Ejecutar las celdas en orden

## Celda 1 — Verificar GPU

In [ ]:
!nvidia-smi

## Celda 2 — Instalar YOLOv8

In [ ]:
!pip install ultralytics -q

## Celda 3 — Montar Drive y descomprimir EPP_merged_v2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_NAME     = 'EPP_merged_v2.zip'
ZIP_PATH     = f'/content/drive/MyDrive/EPP_Dataset/{ZIP_NAME}'
EXTRACT_PATH = '/content/dataset'

print(f'Descomprimiendo {ZIP_NAME}...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for member in z.infolist():
        member.filename = member.filename.replace('\\', '/')
        z.extract(member, EXTRACT_PATH)

print('Listo. Contenido:')
print(os.listdir(EXTRACT_PATH))

## Celda 4 — Verificar distribución de clases

Contamos cuántas anotaciones hay por clase para confirmar que los datos de Safety Vests se fusionaron bien.

In [ ]:
import glob
from collections import Counter

DATASET_DIR = '/content/dataset/EPP_merged_v2'
nombres = ['helmet', 'no-helmet', 'vest', 'no-vest']

for split in ['train', 'valid', 'test']:
    conteo = Counter()
    labels_dir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(labels_dir):
        print(f'[{split}] no encontrado')
        continue
    for fpath in glob.glob(os.path.join(labels_dir, '*.txt')):
        with open(fpath) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    conteo[int(parts[0])] += 1
    print(f'\n[{split}]')
    for cid, name in enumerate(nombres):
        print(f'  {name:<12}: {conteo.get(cid, 0):>7,}')

## Celda 5 — Crear data.yaml

In [ ]:
DATASET_DIR = '/content/dataset/EPP_merged_v2'
YAML_PATH   = os.path.join(DATASET_DIR, 'data.yaml')

yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
test: {DATASET_DIR}/test/images

nc: 4
names: ['helmet', 'no-helmet', 'vest', 'no-vest']
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('data.yaml creado:')
print(yaml_content)

## Celda 6 — Fine-tuning desde best.pt

Partimos del modelo ya entrenado (`modelo_epp_v3/best.pt`) en lugar de `yolov8n.pt`.
Esto permite aprender las nuevas imágenes de chaleco sin olvidar lo que ya sabe de cascos.

- **epochs=30** — suficiente para adaptar sin sobreentrenar
- **lr0=0.001** — learning rate bajo para fine-tuning (10x menor que entrenamiento desde cero)
- **freeze=10** — congela las primeras 10 capas del backbone para preservar características generales

In [ ]:
from ultralytics import YOLO

BASE_MODEL = '/content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/weights/best.pt'

model = YOLO(BASE_MODEL)

results = model.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=8,
    lr0=0.001,
    freeze=10,
    device=0,
    project='/content/drive/MyDrive/EPP_Dataset',
    name='modelo_epp_v4',
    exist_ok=True
)

print('\n=== FINE-TUNING COMPLETADO ===')
print('Mejor modelo: /content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/weights/best.pt')

## Celda 7 — (SOLO SI SE DESCONECTÓ) Reanudar fine-tuning

Si Colab se desconectó, ejecuta esta celda en lugar de la Celda 6.

In [ ]:
# ---- SOLO SI COLAB SE DESCONECTÓ ----
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/weights/last.pt')
results = model.train(resume=True)

print('Entrenamiento reanudado y completado.')

## Celda 8 — Ver gráficas del entrenamiento

In [ ]:
from IPython.display import Image as IPImage
import glob

for img_path in sorted(glob.glob('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/*.png')):
    print(img_path)
    display(IPImage(img_path, width=800))

## Celda 9 — Métricas por clase (mAP50)

Verificar especialmente `no-vest` — debería superar 60% con el nuevo dataset.

In [ ]:
from ultralytics import YOLO

model   = YOLO('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/weights/best.pt')
metrics = model.val(data=YAML_PATH, device=0)

names = ['helmet', 'no-helmet', 'vest', 'no-vest']
print(f'\n{"Clase":<15} {"mAP50":>8}')
print('-' * 25)
for i, name in enumerate(names):
    try:
        print(f'{name:<15} {metrics.box.maps[i]:>8.1%}')
    except Exception:
        print(f'{name:<15} {"N/A":>8}')
print(f'{"PROMEDIO":<15} {metrics.box.map50:>8.1%}')

## Celda 10 — Validar con imágenes de prueba

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage
import glob, random, os

model = YOLO('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/weights/best.pt')

# Probar con imágenes del dataset Safety Vests (prefijo sv_)
test_imgs = glob.glob(f'{DATASET_DIR}/test/images/sv_*.jpg')
if not test_imgs:
    test_imgs = glob.glob(f'{DATASET_DIR}/test/images/*.jpg')
muestra = random.sample(test_imgs, min(6, len(test_imgs)))

os.makedirs('/content/predicciones', exist_ok=True)
for img_path in muestra:
    r = model(img_path, conf=0.4)
    r[0].save(filename=f'/content/predicciones/{os.path.basename(img_path)}')

for pred_img in glob.glob('/content/predicciones/*.jpg'):
    display(IPImage(pred_img, width=640))

## Celda 11 — Descargar modelo_epp_v4

Descargarlo y reemplazar `best.pt` en la carpeta del proyecto (junto a `MonitorEPP.exe`).

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v4/weights/best.pt')